[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [sqlite3, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlite3-deep-dive.html)

# Tables and Queries


## What you will be able to do

Design tables for the stations and their readings, with a key in each and a column in one that
points at the other, create them with `CREATE TABLE`, and find out what a database file you did not
create holds. Write `SELECT` queries that filter, sort, page, summarize and join, quote text and
names the way SQLite expects, and recognize the misspelled column that SQLite quietly turned into
text.


## The idea

### The problem

The table of readings in the **Why sqlite3** notebook repeats a station's name in every row: `Bergen`
8,760 times, `Oslo` 8,760 times, and every question about a station matches on that text. That works
until the stations need more than a name. Put a station's latitude in the same table and the latitude
is copied into 8,760 rows, a corrected latitude has to be corrected in all of them, and nothing stops
one copy from disagreeing with the rest. A station that has just joined the network has nowhere to
live at all, since every row in that table is a reading, and the new station has sent none.

The way out is a table for each kind of thing, stations in one and readings in the other, with every
reading pointing at its station by a number. That raises the questions this notebook answers: how to
create tables that fit together, how to ask a question that needs both of them, and how to write SQL
that SQLite reads the way you meant it, which, for one kind of quotation mark, it does not always do.

### What a table and a query are

> A **table** is a named set of **rows** that share the same **columns**, made by a `CREATE TABLE`
> statement that names every column and can give each one a type and rules. A **primary key** is a
> column whose value identifies one row, and in SQLite a column declared `INTEGER PRIMARY KEY` is
> filled with the next number whenever an insert leaves it out. A column that holds another table's
> primary key, such as a reading's `station_id`, is a **foreign key**, and a **join** matches rows
> of the two tables through it. A **query** is a `SELECT` statement: it names the columns it
> returns, the tables it reads with `FROM` and `JOIN`, the rows it keeps with `WHERE`, how it groups
> them with `GROUP BY` and which groups it keeps with `HAVING`, and the order and number of rows it
> returns with `ORDER BY` and `LIMIT`. In SQL, text goes in single quotes, as in `'Oslo'`, and the
> name of a table or a column may go in double quotes, as in `"station_id"`.

### Why it works that way

- **One fact lives in one place.** A station's latitude is in one row of `stations`, so a correction
  is one `UPDATE`, and a reading refers to its station by `station_id` rather than copying anything
  about it.
- **An id is small and never changes.** A name can be corrected, while a station keeps its id, and
  an integer in 35,040 readings takes less room than a name repeated in all of them.
- **SQL describes the result, not the steps.** A query names the rows it wants, and SQLite works out
  how to find them. The clauses of a `SELECT` are written in a fixed order, which is not the order
  they take effect in: `FROM` and `WHERE` first, then `GROUP BY` and `HAVING`, then the columns the
  query returns, and `ORDER BY` and `LIMIT` last.
- **`WHERE` filters rows before they are grouped, and `HAVING` filters the groups.** A condition on
  an average belongs in `HAVING`, because while `WHERE` runs there is no average yet.
- **A name two joined tables share is ambiguous.** Once `stations` and `readings` are joined, `id`
  could mean either table's column, so SQLite makes you say which, as `stations.id` or `readings.id`.
- **Double quotes are for names, and SQLite bends the rule.** When a word in double quotes matches
  no column, SQLite treats it as text instead of raising an error, a leniency its own documentation
  lists among its quirks. A misspelled column in double quotes becomes that word, repeated in every
  row.
- **A database file can describe itself.** `sqlite_schema` lists every table in a file with the
  `CREATE TABLE` statement that made it, and `PRAGMA table_info` lists the columns of one table.

### Where this shows up

Every relational database keeps its data in tables linked by keys and reads it with `SELECT`, so
what this notebook writes carries over. PostgreSQL, in the **asyncpg and psycopg3, Deep Dive**
guide, speaks the same SQL with a few differences, and the **SQLAlchemy, Deep Dive** guide generates
these statements from Python classes. In the **Pandas, Deep Dive** guide, `merge` is a join by
another name and `groupby` does the work of `GROUP BY`. The **DuckDB, Deep Dive** guide runs the
same kind of query over CSV and Parquet files, with far more for summarizing than this notebook
needs. Falling back from a name to text is SQLite's habit, not SQL's: PostgreSQL, for one, raises an
error for the misspelled column.

### What this notebook covers

- Two tables that fit together, created with `executescript`
- Rows, and the ids SQLite gives them
- What a database file holds: `sqlite_schema`, `PRAGMA table_info`, and `CREATE TABLE IF NOT EXISTS`
- Picking rows with `WHERE`: comparisons, `AND`, `IN`, `BETWEEN`, `LIKE` and `IS NULL`
- Sorting and paging with `ORDER BY`, `LIMIT` and `OFFSET`, and `DISTINCT`
- Columns a query works out for itself, with `AS`, `ROUND` and `CASE`
- Groups, with `GROUP BY`, `COUNT`, `AVG` and `HAVING`
- Joining the two tables, with an inner join and a left join
- Single quotes and double quotes
- A frost report for the stations north of a latitude, one of them with no readings yet
- Seven errors: a comma before `FROM`, a misspelled column, a column two joined tables share, an
  average in `WHERE`, two statements in one `execute`, a misspelled column in double quotes, and a
  left join undone by its `WHERE`

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read
it, and read the output underneath it. Everything from Setup onward is where you start running
things, and the rest of the notebook takes this apart piece by piece.

```python
import sqlite3

conn = sqlite3.connect(":memory:")
conn.executescript("""
    CREATE TABLE stations (id INTEGER PRIMARY KEY, name TEXT NOT NULL, latitude REAL NOT NULL);
    CREATE TABLE readings (id INTEGER PRIMARY KEY, station_id INTEGER NOT NULL, celsius REAL);
    INSERT INTO stations (name, latitude)
        VALUES ('Bergen', 60.39), ('Svalbard', 78.22), ('Kirkenes', 69.73);
    INSERT INTO readings (station_id, celsius) VALUES (1, 4.2), (1, 3.9), (2, -18.5), (2, -19.1);
""")

query = """
    SELECT stations.name, COUNT(readings.id), MIN(readings.celsius)
    FROM stations LEFT JOIN readings ON readings.station_id = stations.id
    GROUP BY stations.id
    ORDER BY stations.latitude DESC
"""
for row in conn.execute(query):
    print(row)
conn.close()
```

```
('Svalbard', 2, -19.1)
('Kirkenes', 0, None)
('Bergen', 2, 3.9)
```

Two tables, each with its own key, and one query that reads both: every station, how many readings
it has, and its coldest one, from north to south. Kirkenes has no readings yet, and the left join
kept it anyway, with a count of 0 and no coldest reading. The name came from one table and the
readings from the other, matched through `station_id`.


## Setup

Six imports, the year of readings as a generator, and the latitudes of five stations. The worked
examples build the tables themselves, since designing them is what this notebook is about.

- `sqlite3` connects to the database and runs every statement
- `closing`, from `contextlib`, closes a second connection that reads the database, in the worked
  examples
- `math`, `datetime` and `timedelta` make the same year of readings the **Why sqlite3** notebook made
- `Path` names the scratch folder and the database in it
- `shutil` removes the scratch folder at the end

`LATITUDES` has one station more than `STATIONS`: Kirkenes, which joined the network too recently to
have sent a reading.


In [1]:
import math
import shutil
import sqlite3
from contextlib import closing
from datetime import datetime, timedelta
from pathlib import Path

SCRATCH = Path("scratch")
SCRATCH.mkdir(exist_ok=True)
DATABASE = SCRATCH / "stations.db"
DATABASE.unlink(missing_ok=True)
STATIONS = {"Bergen": 8.0, "Oslo": 6.5, "Svalbard": -4.5, "Tromso": 3.5}   # each station's mean for the year
LATITUDES = {"Bergen": 60.39, "Oslo": 59.91, "Svalbard": 78.22, "Tromso": 69.65, "Kirkenes": 69.73}


def year_of_readings():
    """Every hour of 2025 at the four stations, as Why sqlite3 made them, with Svalbard silent on 2 March."""
    for n in range(365 * 24):
        hour = datetime(2025, 1, 1) + timedelta(hours=n)
        season = -math.cos(2 * math.pi * (n - 400) / (365 * 24))
        day = -math.cos(2 * math.pi * (hour.hour - 3) / 24)
        for i, (station, mean) in enumerate(STATIONS.items()):
            if station == "Svalbard" and hour.strftime("%Y-%m-%d") == "2025-03-02":
                celsius = None
            else:
                wobble = ((n * 37 + i * 101) % 17 - 8) / 10
                celsius = round(mean + 9 * season + 3 * day + wobble, 1) + 0.0
            yield station, hour.strftime("%Y-%m-%dT%H:%M"), celsius


print("ready to build", DATABASE, "for", len(LATITUDES), "stations")


ready to build scratch/stations.db for 5 stations


## Worked examples

### Two tables that fit together

A station is one kind of thing and a reading is another, so each gets a table. `stations` has a row
for every station, identified by `id INTEGER PRIMARY KEY`. `readings` has a row for every reading,
and its `station_id` holds the id of the station that sent it. `REFERENCES stations (id)` records
which column that is, a rule the **Constraints** notebook makes SQLite enforce. `execute` runs one
statement at a time, and `executescript` runs several, separated by semicolons, so it creates both
tables in one call:


In [2]:
conn = sqlite3.connect(DATABASE)
conn.executescript("""
    CREATE TABLE stations (
        id       INTEGER PRIMARY KEY,
        name     TEXT NOT NULL,
        latitude REAL NOT NULL
    );

    CREATE TABLE readings (
        id         INTEGER PRIMARY KEY,
        station_id INTEGER NOT NULL REFERENCES stations (id),
        hour       TEXT NOT NULL,
        celsius    REAL
    );
""")

print("stations columns:", [column[0] for column in conn.execute("SELECT * FROM stations").description])
print("readings columns:", [column[0] for column in conn.execute("SELECT * FROM readings").description])


stations columns: ['id', 'name', 'latitude']
readings columns: ['id', 'station_id', 'hour', 'celsius']


`TEXT`, `INTEGER` and `REAL` declare what a column is meant to hold, and `NOT NULL` refuses a row
without a value. How strictly SQLite keeps to a declared type is a story of its own, which the **Type
Affinity** notebook tells.

### Rows, and the ids SQLite gives them

An insert into `stations` leaves out `id`, and SQLite fills it with the next number. The cursor's
`lastrowid` reports the number it used. The ids go into a dictionary, and every reading goes in with
its station's id in place of the station's name:


In [3]:
station_ids = {}
for name, latitude in LATITUDES.items():
    cursor = conn.execute("INSERT INTO stations (name, latitude) VALUES (?, ?)", (name, latitude))
    station_ids[name] = cursor.lastrowid
print("ids:", station_ids)

conn.executemany("INSERT INTO readings (station_id, hour, celsius) VALUES (?, ?, ?)",
                 ((station_ids[station], hour, celsius) for station, hour, celsius in year_of_readings()))
conn.commit()
print("readings:", conn.execute("SELECT COUNT(*) FROM readings").fetchone()[0])


ids: {'Bergen': 1, 'Oslo': 2, 'Svalbard': 3, 'Tromso': 4, 'Kirkenes': 5}
readings: 35040


Kirkenes has an id and no readings, which the joins below make use of.

### What a database file holds

A database file you did not create can tell you what is in it. `sqlite_schema` is a table SQLite
keeps in every database, with a row for every table and the `CREATE TABLE` statement that made it.
`PRAGMA table_info` lists one table's columns, with the type each one declares, whether it refuses
`NULL`, and whether it is the primary key. A second connection asks, as a program that had only the
file would:


In [4]:
with closing(sqlite3.connect(DATABASE)) as other:
    for kind, name in other.execute("SELECT type, name FROM sqlite_schema ORDER BY name"):
        print(kind, name)
    for cid, name, declared, notnull, default, key in other.execute("PRAGMA table_info(readings)"):
        print(f"  {name:<11} {declared:<8} not null: {bool(notnull)!s:<6} primary key: {bool(key)}")


table readings
table stations
  id          INTEGER  not null: False  primary key: True
  station_id  INTEGER  not null: True   primary key: False
  hour        TEXT     not null: True   primary key: False
  celsius     REAL     not null: False  primary key: False


`not null` reports only what a column declares, so `id` shows False, although an `INTEGER PRIMARY
KEY` is never empty: an insert that leaves it out, or gives it `NULL`, gets the next number instead.
Older code, and many examples you will find, call `sqlite_schema` by its old name, `sqlite_master`,
which still works.

`CREATE TABLE` refuses a table that already exists. `CREATE TABLE IF NOT EXISTS` creates a table only
when the file has none of that name, so a program can run its `CREATE TABLE` statements every time it
starts. It checks the name and nothing else: a table that exists with different columns is left as it
is, which the **Changing a Schema** notebook deals with:


In [5]:
try:
    conn.execute("CREATE TABLE stations (id INTEGER PRIMARY KEY, name TEXT NOT NULL, latitude REAL NOT NULL)")
except sqlite3.OperationalError as error:
    print("CREATE TABLE:", error)

conn.execute("CREATE TABLE IF NOT EXISTS stations (id INTEGER PRIMARY KEY, name TEXT NOT NULL, latitude REAL NOT NULL)")
rows = conn.execute("SELECT COUNT(*) FROM stations").fetchone()[0]
print("CREATE TABLE IF NOT EXISTS: stations still holds", rows, "rows")


CREATE TABLE: table stations already exists
CREATE TABLE IF NOT EXISTS: stations still holds 5 rows


### Picking rows with WHERE

`WHERE` keeps the rows its condition is true for. Conditions combine with `AND` and `OR`, `IN` tests
against a list of values, `BETWEEN` tests against a range that includes both ends, and `LIKE` matches
a pattern in which `%` stands for any run of characters. Every value here goes in through a
placeholder, as in the **Why sqlite3** notebook:


In [6]:
oslo, bergen = station_ids["Oslo"], station_ids["Bergen"]

hours = conn.execute("SELECT COUNT(*) FROM readings WHERE station_id = ? AND celsius < ?",
                     (oslo, -5)).fetchone()[0]
print("hours at Oslo below -5:", hours)

hours = conn.execute("SELECT COUNT(*) FROM readings WHERE station_id = ? AND celsius BETWEEN ? AND ?",
                     (oslo, -1, 1)).fetchone()[0]
print("hours at Oslo from -1 to 1:", hours)

rows = conn.execute("SELECT COUNT(*) FROM readings WHERE station_id IN (?, ?) AND hour LIKE ?",
                    (bergen, oslo, "2025-06-21%")).fetchone()[0]
print("readings on midsummer's day at Bergen and Oslo:", rows)


hours at Oslo below -5: 134
hours at Oslo from -1 to 1: 1017
readings on midsummer's day at Bergen and Oslo: 48


`NULL` needs a test of its own. `IS NULL` finds a missing reading, and `= NULL` finds nothing, not
even a `NULL`, because a comparison with `NULL` is neither true nor false. And `=` compares text
exactly, while `LIKE` ignores the difference between capital and small letters, at least for the
letters of English:


In [7]:
print("celsius IS NULL: ", conn.execute("SELECT COUNT(*) FROM readings WHERE celsius IS NULL").fetchone()[0])
print("celsius = NULL:  ", conn.execute("SELECT COUNT(*) FROM readings WHERE celsius = NULL").fetchone()[0])
print("name = 'oslo':   ", conn.execute("SELECT name FROM stations WHERE name = 'oslo'").fetchall())
print("name LIKE 'oslo':", conn.execute("SELECT name FROM stations WHERE name LIKE 'oslo'").fetchall())


celsius IS NULL:  24
celsius = NULL:   0
name = 'oslo':    []
name LIKE 'oslo': [('Oslo',)]


### Sorting and paging with ORDER BY, LIMIT and OFFSET

`ORDER BY` sorts by one column and then by the next where the first is equal, and `DESC` reverses a
column's order. `LIMIT` keeps a number of rows, and `OFFSET` skips a number first, which is how an
application shows results a page at a time. `DISTINCT` keeps one copy of every row that repeats:


In [8]:
coldest = """
    SELECT station_id, hour, celsius FROM readings
    WHERE celsius IS NOT NULL
    ORDER BY celsius, hour
    LIMIT 3 OFFSET ?
"""
print("coldest, page 1:", conn.execute(coldest, (0,)).fetchall())
print("coldest, page 2:", conn.execute(coldest, (3,)).fetchall())
print("stations that went below -5:",
      conn.execute("SELECT DISTINCT station_id FROM readings WHERE celsius < -5 ORDER BY station_id").fetchall())


coldest, page 1: [(3, '2025-01-12T03:00', -17.3), (3, '2025-01-17T02:00', -17.2), (3, '2025-01-07T04:00', -17.1)]
coldest, page 2: [(3, '2025-01-08T03:00', -17.1), (3, '2025-01-13T02:00', -17.1), (3, '2025-01-20T04:00', -17.1)]
stations that went below -5: [(2,), (3,), (4,)]


The readings of -17.1 run across both pages, and `hour` decides which of them come first. Sorted by
`celsius` alone, readings with the same temperature can come back in any order, so a request for the
second page could repeat a reading from the first, or skip one. `DISTINCT` turned the thousands of
readings below -5 into the three stations that sent them.

### Columns a query works out for itself

A query can return values that no table holds: an expression worked out from columns, named with
`AS`, or a choice made by `CASE`, which works like `if` and `else`. Here is the morning of 1 March at
Oslo, with every reading in Fahrenheit and marked for frost. `BETWEEN` works on the hours because
text written this way sorts in time order:


In [9]:
morning = conn.execute("""
    SELECT hour,
           celsius,
           ROUND(celsius * 9 / 5 + 32, 1) AS fahrenheit,
           CASE WHEN celsius < 0 THEN 'frost' ELSE 'no frost' END AS frost
    FROM readings
    WHERE station_id = ? AND hour BETWEEN ? AND ?
    ORDER BY hour
""", (oslo, "2025-03-01T06:00", "2025-03-01T11:00"))

print("columns:", [column[0] for column in morning.description])
for row in morning:
    print(row)


columns: ['hour', 'celsius', 'fahrenheit', 'frost']
('2025-03-01T06:00', -1.6, 29.1, 'frost')
('2025-03-01T07:00', -2.4, 27.7, 'frost')
('2025-03-01T08:00', -1.4, 29.5, 'frost')
('2025-03-01T09:00', -0.3, 31.5, 'frost')
('2025-03-01T10:00', 0.8, 33.4, 'no frost')
('2025-03-01T11:00', 1.8, 35.2, 'no frost')


`AS` names a column in the result, and `description` reports that name, so it reaches whatever reads
the rows. `ROUND(..., 1)` keeps the one decimal place the readings have, and `CASE` returns the value
of the first `WHEN` that is true, or of `ELSE` when none is.

### Groups, with GROUP BY, COUNT and HAVING

`GROUP BY` collects the rows that share a value into a group, and an aggregate such as `COUNT` or
`AVG` turns every group into one value. `HAVING` then keeps the groups its condition is true for.
Here are the stations whose mean for the year was below 5 degrees:


In [10]:
cold_stations = conn.execute("""
    SELECT station_id, COUNT(*) AS hours, COUNT(celsius) AS readings, AVG(celsius) AS mean
    FROM readings
    GROUP BY station_id
    HAVING AVG(celsius) < 5
    ORDER BY mean
""")

for station_id, hours, readings, mean in cold_stations:
    print(f"station {station_id}: {hours} hours, {readings} readings, mean {mean:.1f}")


station 3: 8760 hours, 8736 readings, mean -4.5
station 4: 8760 hours, 8760 readings, mean 3.5


`COUNT(*)` counts rows and `COUNT(celsius)` counts the readings that are not `NULL`, so station 3,
Svalbard, is 24 readings short: 2 March, the day it sent nothing. `WHERE` could not have asked this
question, since it runs on single readings before they are grouped, when there is no mean to test.
The result names stations by id, which is where a join comes in. Summaries past a count and a mean
are the **DuckDB, Deep Dive** guide's subject.

### Joining the two tables

A join puts a row of one table beside the matching rows of the other. `JOIN ... ON` names the
condition that matches them, here a reading's `station_id` against a station's `id`. `AS` gives each
table a short name, and a column named with its table, as in `s.name`, says which table it comes
from. Every station's hours below freezing, from north to south:


In [11]:
freezing = conn.execute("""
    SELECT s.name, s.latitude, COUNT(r.id) AS hours
    FROM stations AS s
    JOIN readings AS r ON r.station_id = s.id
    WHERE r.celsius < 0
    GROUP BY s.id
    ORDER BY s.latitude DESC
""")

for name, latitude, hours in freezing:
    print(f"{name:<9} {latitude:6.2f}  {hours:>5} hours below freezing")


Svalbard   78.22   5871 hours below freezing
Tromso     69.65   3203 hours below freezing
Bergen     60.39   1220 hours below freezing
Oslo       59.91   1848 hours below freezing


Kirkenes is missing. A plain `JOIN` keeps only rows that found a match, and Kirkenes has no readings
to match. `LEFT JOIN` keeps every row of the table on its left, and where a station has no readings
it fills the reading's columns with `NULL`. `COUNT(r.id)` counts only rows with a reading, so
Kirkenes counts 0, while `COUNT(*)` counts every row, including Kirkenes's one row of `NULL`:


In [12]:
per_station = conn.execute("""
    SELECT s.name, COUNT(r.id) AS readings, COUNT(*) AS rows
    FROM stations AS s
    LEFT JOIN readings AS r ON r.station_id = s.id
    GROUP BY s.id
    ORDER BY s.name
""")

for name, readings, rows in per_station:
    print(f"{name:<9} COUNT(r.id) = {readings:<5} COUNT(*) = {rows}")


Bergen    COUNT(r.id) = 8760  COUNT(*) = 8760
Kirkenes  COUNT(r.id) = 0     COUNT(*) = 1
Oslo      COUNT(r.id) = 8760  COUNT(*) = 8760
Svalbard  COUNT(r.id) = 8760  COUNT(*) = 8760
Tromso    COUNT(r.id) = 8760  COUNT(*) = 8760


### Single quotes and double quotes

SQL uses its two quotation marks for two different things. Single quotes make text, so `'name'` is
the word name. Double quotes make a name, so `"name"` is the column, and they let a name contain a
space or be a word SQL keeps for itself, such as `"order"`. Python writes its strings in either,
which is exactly the habit not to carry into SQL. One query shows the difference:


In [13]:
query = """SELECT 'name', "name", 'latitude', "latitude" FROM stations WHERE "name" = 'Svalbard'"""

print(conn.execute(query).fetchone())


('name', 'Svalbard', 'latitude', 78.22)


Single quotes gave back the words themselves, and double quotes gave back Svalbard's values. The rule
is simple: text in single quotes, names bare or in double quotes. SQLite does not hold you to it,
which one of the Common errors below shows.

### A frost report from both tables

The pieces of this notebook in one job: every station north of a latitude, with its hours below
freezing and the month it froze most, including a station that has no readings yet. In the first
query, `WHERE` tests the station's own latitude, which decides the stations that appear. The frost
condition is about the readings, so it sits in the left join's `ON`, where a station without frost
keeps its row. The second query runs once for every station the first one returned, and
`substr(hour, 1, 7)` takes the first seven characters of an hour, such as `2025-01`, its month:


In [14]:
def frost_report(conn, north_of):
    """Stations north of a latitude, their hours below freezing, and each one's month with the most frost."""
    stations = conn.execute("""
        SELECT s.id, s.name, s.latitude, COUNT(r.id) AS frost_hours
        FROM stations AS s
        LEFT JOIN readings AS r ON r.station_id = s.id AND r.celsius < 0
        WHERE s.latitude > ?
        GROUP BY s.id
        ORDER BY s.latitude DESC
    """, (north_of,)).fetchall()

    for station_id, name, latitude, frost_hours in stations:
        worst = conn.execute("""
            SELECT substr(hour, 1, 7) AS month, COUNT(*) AS hours
            FROM readings
            WHERE station_id = ? AND celsius < 0
            GROUP BY month
            ORDER BY hours DESC, month
            LIMIT 1
        """, (station_id,)).fetchone()
        month = f"worst month {worst[0]} with {worst[1]} hours" if worst else "no readings yet"
        print(f"{name:<9} {latitude:6.2f}  {frost_hours:>5} hours below freezing, {month}")


frost_report(conn, 60)


Svalbard   78.22   5871 hours below freezing, worst month 2025-01 with 744 hours
Kirkenes   69.73      0 hours below freezing, no readings yet
Tromso     69.65   3203 hours below freezing, worst month 2025-01 with 744 hours
Bergen     60.39   1220 hours below freezing, worst month 2025-01 with 441 hours


Four stations north of 60 degrees, from north to south, with Oslo left out by `WHERE`. Svalbard
froze for all 744 hours of both January and December, and `month` in `ORDER BY` broke the tie.
Kirkenes kept its row with 0 hours, and its second query found no row at all, so `fetchone` returned
`None`.

### Where each part came from

| In the report | What it relies on | The section that showed it |
|---|---|---|
| `stations` and `readings`, joined on `station_id` | a table for each kind of thing, linked by a key | Two tables that fit together |
| `s.id` and `r.station_id` | ids that SQLite filled in, stored in place of names | Rows, and the ids SQLite gives them |
| `WHERE s.latitude > ?` and `celsius < 0` | conditions that keep some rows and not others | Picking rows with WHERE |
| `ORDER BY hours DESC, month` with `LIMIT 1` | a sort with a tie-breaker, and one row kept | Sorting and paging with ORDER BY, LIMIT and OFFSET |
| `substr(hour, 1, 7) AS month` | a column the query works out and names | Columns a query works out for itself |
| `GROUP BY month` and `COUNT(*)` | rows collected into groups, one value for each | Groups, with GROUP BY, COUNT and HAVING |
| `LEFT JOIN ... ON ... AND r.celsius < 0` | every station kept, with only its frost joined | Joining the two tables |


## Your turn

Six tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than getting
there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlite3-deep-dive/03-tables-and-queries-solutions.ipynb).

**1.** Print every station's name and latitude, from north to south.


In [15]:
# your code here


**2.** For every station that ever went above 15 degrees, print its name and the number of hours it
spent above 15, finding the name through a join.


In [16]:
# your code here


**3.** Print Tromso's three warmest readings with their hours, finding Tromso by its name in the
query, not by its id.


In [17]:
# your code here


**4.** Print the name of every station that has no readings at all.


In [18]:
# your code here


**5.** Print the months in which Svalbard's mean was below -8 degrees, with the mean to one decimal
place, using `HAVING`.


In [19]:
# your code here


**6.** From `sqlite_schema`, print the `CREATE TABLE` statement that SQLite keeps for `readings`.


In [20]:
# your code here


## Common errors

### sqlite3.OperationalError: near "FROM": syntax error


In [21]:
conn.execute("""
    SELECT name,
           latitude,
    FROM stations
    ORDER BY latitude
""")


OperationalError: near "FROM": syntax error

SQLite stopped at `FROM`, but the mistake is the comma before it: after a comma SQLite expects
another column, and `FROM` cannot be one. A syntax error names the word where SQLite could no longer
make sense of the statement, so look at that word and at what comes just before it. The last column
in the list takes no comma:


In [22]:
print(conn.execute("""
    SELECT name,
           latitude
    FROM stations
    ORDER BY latitude
""").fetchall())


[('Oslo', 59.91), ('Bergen', 60.39), ('Tromso', 69.65), ('Kirkenes', 69.73), ('Svalbard', 78.22)]


### sqlite3.OperationalError: no such column: celcius


In [23]:
conn.execute("SELECT hour, celcius FROM readings WHERE station_id = ?", (oslo,))


OperationalError: no such column: celcius

`celcius` is not a column of `readings`, and SQLite checks every name in a statement before it runs
any of it. The message names the word it could not find, which is usually a spelling. `PRAGMA
table_info` lists the real names when you are not sure of them:


In [24]:
print([name for cid, name, *rest in conn.execute("PRAGMA table_info(readings)")])
print(conn.execute("SELECT hour, celsius FROM readings WHERE station_id = ? ORDER BY hour LIMIT 2", (oslo,)).fetchall())


['id', 'station_id', 'hour', 'celsius']
[('2025-01-01T00:00', -3.5), ('2025-01-01T01:00', -5.3)]


### sqlite3.OperationalError: ambiguous column name: id


In [25]:
conn.execute("""
    SELECT id, name, hour, celsius
    FROM stations JOIN readings ON station_id = stations.id
    LIMIT 2
""")


OperationalError: ambiguous column name: id

Both tables have a column called `id`, and once they are joined SQLite cannot tell which one the
query means, so it refuses rather than guess. `name`, `hour` and `celsius` are in only one of the
tables, so they needed no qualifying. Name the table for a column the joined tables share, and give
the columns distinct names in the result with `AS` if both are wanted:


In [26]:
print(conn.execute("""
    SELECT s.id AS station, r.id AS reading, s.name, r.hour, r.celsius
    FROM stations AS s JOIN readings AS r ON r.station_id = s.id
    ORDER BY r.id
    LIMIT 2
""").fetchall())


[(1, 1, 'Bergen', '2025-01-01T00:00', -3.6), (2, 2, 'Oslo', '2025-01-01T00:00', -3.5)]


### sqlite3.OperationalError: misuse of aggregate: AVG()


In [27]:
conn.execute("""
    SELECT station_id, AVG(celsius) AS mean
    FROM readings
    WHERE AVG(celsius) < 5
    GROUP BY station_id
""")


OperationalError: misuse of aggregate: AVG()

`WHERE` tests one row at a time, before the rows are grouped, and an average only exists once a group
does, so SQLite refuses an aggregate in `WHERE`. A condition on a group belongs in `HAVING`, which
takes effect after `GROUP BY`:


In [28]:
print(conn.execute("""
    SELECT station_id, ROUND(AVG(celsius), 1) AS mean
    FROM readings
    GROUP BY station_id
    HAVING AVG(celsius) < 5
    ORDER BY station_id
""").fetchall())


[(3, -4.5), (4, 3.5)]


### sqlite3.ProgrammingError: You can only execute one statement at a time.


In [29]:
conn.execute("""
    CREATE TABLE visits (station_id INTEGER NOT NULL, day TEXT NOT NULL);
    INSERT INTO visits (station_id, day) VALUES (1, '2025-05-12');
""")


ProgrammingError: You can only execute one statement at a time.

`execute` runs one statement, and it found a second after the first semicolon. It raised before
running either of them, so no `visits` table exists. Run a script of several statements, with no
values to fill in, through `executescript`, or give every statement an `execute` of its own, which is
the way to go whenever a statement needs a placeholder:


In [30]:
print("visits exists:", conn.execute("SELECT COUNT(*) FROM sqlite_schema WHERE name = 'visits'").fetchone()[0] == 1)

conn.executescript("""
    CREATE TABLE visits (station_id INTEGER NOT NULL, day TEXT NOT NULL);
    INSERT INTO visits (station_id, day) VALUES (1, '2025-05-12');
""")
print("visits:", conn.execute("SELECT station_id, day FROM visits").fetchall())


visits exists: False
visits: [(1, '2025-05-12')]


### No error, and the word celcius in every row: a misspelled column in double quotes


In [31]:
conn.setconfig(sqlite3.SQLITE_DBCONFIG_DQS_DML, True)     # on in most builds of SQLite, and set here in case

print(conn.execute("""
    SELECT hour, "celcius" FROM readings WHERE station_id = ? ORDER BY hour LIMIT 3
""", (oslo,)).fetchall())
print("averaged:", conn.execute('SELECT AVG("celcius"), COUNT("celcius") FROM readings').fetchone())


[('2025-01-01T00:00', 'celcius'), ('2025-01-01T01:00', 'celcius'), ('2025-01-01T02:00', 'celcius')]
averaged: (0.0, 35040)


The same misspelling as in `no such column: celcius`, now in double quotes, and this time no error
at all. No column is called `celcius`, so SQLite read `"celcius"` as the text it would have been in
single quotes, and returned that word in place of every reading. `AVG` over that column returns 0.0,
over all 35,040 rows, since text that does not look like a number counts as 0, so a report can show a
mean of zero degrees rather than an error. SQLite's documentation lists this fallback
among its quirks, kept so that old programs go on working, and a build of SQLite can be compiled
without it, which is why the cell turns the fallback on first. SQLite's own command-line shell has
refused double-quoted text since SQLite 3.41, so a query that runs quietly in Python can fail there.
Keep text in single quotes, and on Python 3.12 or later tell the connection to refuse double-quoted
text, which turns the misspelling back into an error:


In [32]:
conn.setconfig(sqlite3.SQLITE_DBCONFIG_DQS_DML, False)

try:
    conn.execute('SELECT hour, "celcius" FROM readings LIMIT 3')
except sqlite3.OperationalError as error:
    print("OperationalError:", str(error).split(":")[0])
print(conn.execute('SELECT hour, "celsius" FROM readings WHERE station_id = ? ORDER BY hour LIMIT 3', (oslo,)).fetchall())


OperationalError: no such column
[('2025-01-01T00:00', -3.5), ('2025-01-01T01:00', -5.3), ('2025-01-01T02:00', -5.3)]


With the setting off, the misspelled name is an error again, and the correct name still works in
double quotes, since double quotes around a real name were never the problem. From SQLite 3.46 the
message goes on to ask whether the word should be a string literal in single quotes, while earlier
versions stop after the name, so the cell prints only the part every version shares.

### No error, and stations missing from a left join: a WHERE on the joined table


In [33]:
cold_hours = conn.execute("""
    SELECT s.name, COUNT(r.id) AS hours
    FROM stations AS s
    LEFT JOIN readings AS r ON r.station_id = s.id
    WHERE r.celsius < -5
    GROUP BY s.id
    ORDER BY s.name
""")

print(cold_hours.fetchall())


[('Oslo', 134), ('Svalbard', 4179), ('Tromso', 1041)]


The left join was meant to keep every station, and Bergen and Kirkenes are missing. `WHERE` takes
effect after the join, on every joined row. Bergen never went below -5 degrees, so none of its rows
passed, and Kirkenes's one row holds `NULL`, which is not below anything, so `WHERE` removed both,
and the left join ended up no different from a plain one. Put a condition on the joined table in
`ON`, where it decides which readings join, not which stations stay:


In [34]:
cold_hours = conn.execute("""
    SELECT s.name, COUNT(r.id) AS hours
    FROM stations AS s
    LEFT JOIN readings AS r ON r.station_id = s.id AND r.celsius < -5
    GROUP BY s.id
    ORDER BY s.name
""")

print(cold_hours.fetchall())
conn.close()


[('Bergen', 0), ('Kirkenes', 0), ('Oslo', 134), ('Svalbard', 4179), ('Tromso', 1041)]


Last, the notebook is finished with its files, so this cell removes the scratch folder, with the
database in it:


In [35]:
shutil.rmtree("scratch")

print("scratch still there:", Path("scratch").exists())


scratch still there: False


## Recap

- A table for each kind of thing, linked by a key, keeps a fact in one place: `INTEGER PRIMARY KEY`
  fills in an id, `lastrowid` reports it, and a column such as `station_id` points at another table.
- `executescript` runs several statements, `CREATE TABLE IF NOT EXISTS` leaves an existing table
  alone, and `sqlite_schema` and `PRAGMA table_info` say what a file holds.
- `WHERE` picks rows with comparisons, `AND`, `IN`, `BETWEEN`, `LIKE` and `IS NULL`, and `= NULL`
  matches nothing.
- `ORDER BY` needs a tie-breaker for `LIMIT` and `OFFSET` to page reliably, and `AS` names a column a
  query works out.
- `COUNT(*)` counts rows where `COUNT(column)` skips `NULL`, and `HAVING` filters the groups that
  `GROUP BY` makes, where `WHERE` filters rows.
- A join matches rows through a key, a left join also keeps rows with no match, and a condition on
  the joined table belongs in `ON`, not `WHERE`.
- A syntax error names the word where SQLite gave up, and the mistake is often just before it.
- Text goes in single quotes and names in double quotes, since SQLite quietly turns a double-quoted
  word that names no column into text unless the connection turns `SQLITE_DBCONFIG_DQS_DML` off.


## What is next

The **SQL Syntax** notebook takes the SQL in these queries further: the order a query is written in
against the order it takes effect in, the kinds of statement SQL has, aggregates past a count, and
dates and times in a `WHERE` clause.


---

&#8592; **Previous:** [Connections and Cursors](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlite3-deep-dive/02-connections-and-cursors.ipynb)  &nbsp;·&nbsp;  [sqlite3, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlite3-deep-dive.html)  &nbsp;·&nbsp;  **Next:** [SQL Syntax](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlite3-deep-dive/04-sql-syntax.ipynb) &#8594;
